# AIC 2026 — OCR keyframes bằng EasyOCR + VietOCR

EasyOCR/CRAFT phát hiện vùng chữ; VietOCR nhận dạng tiếng Việt. Kết quả được lưu theo từng video và có thể tiếp tục sau khi Colab ngắt kết nối.

In [ ]:
!nvidia-smi

# Nguyên tắc: GIỮ NGUYÊN version có sẵn của Colab (torch, numpy, Pillow, opencv…),
# chỉ cài thêm thứ Colab chưa có. Đụng vào chúng là vỡ môi trường.
#
# Hai cái bẫy đã gặp:
#   1. `pip install -U easyocr` — cờ -U upgrade cả DEPENDENCY, kéo torch/numpy/Pillow
#      của Colab lên bản mới. Bỏ -U thì pip thấy dep đã thoả và để nguyên.
#   2. `pip install vietocr` — vietocr 0.3.13 pin `pillow==10.2.0` nên hạ cấp Pillow 11
#      ngay trên cây PIL/ đang dùng, để lại file lẫn version:
#        ImportError: cannot import name 'is_directory' from 'PIL._util'
#      Dùng --no-deps: các dep bị bỏ (pillow, imgaug, albumentations, lmdb,
#      prefetch-generator, scikit-image) chỉ cần khi TRAIN vietocr. Inference chỉ dùng
#      torch / numpy / PIL / yaml / einops / gdown — Colab có sẵn hết trừ einops.
!pip -q install easyocr
!pip -q install --no-deps vietocr
!python -c "import einops" 2>/dev/null || pip -q install --no-deps einops

# Nếu môi trường ĐÃ hỏng từ lần chạy trước, cách sạch nhất là bỏ hẳn session cũ:
# Runtime → Disconnect and delete runtime, rồi mở lại và chạy từ cell này.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
OCR_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/OCR_EasyOCR_VietOCR')
MAP_KEYFRAMES_DIRECTORY = DATASET_DIRECTORY / 'map-keyframes-aic25-b1' / 'map-keyframes'
TARGET_FOLDERS = [
    'Keyframes_L21', 'Keyframes_L22', 'Keyframes_L23', 'Keyframes_L24',
    'Keyframes_L25', 'Keyframes_L26_a', 'Keyframes_L26_b',
    'Keyframes_L26_c', 'Keyframes_L26_d', 'Keyframes_L26_e',
    'Keyframes_L27', 'Keyframes_L28', 'Keyframes_L29', 'Keyframes_L30',
]
MIN_CONFIDENCE = 0.35
BOX_PADDING = 4
MIN_BOX_SIDE = 8
RECOGNITION_BATCH = 32
OVERWRITE = False
MODEL_ID = 'easyocr-craft-det + vietocr-vgg_transformer'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

dataset_root = DATASET_DIRECTORY.resolve()
KEYFRAME_ROOTS, missing = [], []
for folder in TARGET_FOLDERS:
    relative = Path(folder.strip())
    assert not relative.is_absolute(), f'Chỉ nhận đường dẫn tương đối: {folder}'
    selected = (dataset_root / relative).resolve()
    assert selected == dataset_root or dataset_root in selected.parents
    if selected.is_dir(): KEYFRAME_ROOTS.append(selected)
    else: missing.append(folder)
assert KEYFRAME_ROOTS, 'Không tìm thấy thư mục keyframe nào'
assert MAP_KEYFRAMES_DIRECTORY.is_dir(), f'Thiếu map-keyframes: {MAP_KEYFRAMES_DIRECTORY}'
OUTPUT_ROOT = OCR_DIRECTORY.resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if missing: print('Bỏ qua thư mục không tồn tại:', ', '.join(missing))
print('Input:', len(KEYFRAME_ROOTS), '| Output:', OUTPUT_ROOT)

In [ ]:
import csv
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')
def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir(): base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not path.is_file(): return None
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {'pts_time': float(row['pts_time']), 'fps': float(row['fps']), 'frame_idx': int(row['frame_idx'])}
            except (KeyError, TypeError, ValueError): pass
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

video_dirs = sorted((v for root in KEYFRAME_ROOTS for v in find_video_dirs(root)), key=lambda p: p.name)
print(f'Tìm thấy {len(video_dirs)} video; {sum(load_keyframe_map(v.name) is not None for v in video_dirs)} video có map')

## Tải model

In [ ]:
import torch
import easyocr
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

use_gpu = torch.cuda.is_available()
print('Device:', 'GPU' if use_gpu else 'CPU')
detector = easyocr.Reader(['vi'], gpu=use_gpu)
vietocr_config = Cfg.load_config_from_name('vgg_transformer')
vietocr_config['device'] = 'cuda:0' if use_gpu else 'cpu'
vietocr_config['cnn']['pretrained'] = False
recognizer = Predictor(vietocr_config)

In [ ]:
import json
import numpy as np
from PIL import Image

def detect_boxes(image):
    horizontal, free = detector.detect(np.array(image))
    boxes = []
    for x1, x2, y1, y2 in (horizontal[0] if horizontal else []):
        boxes.append((int(x1), int(y1), int(x2), int(y2)))
    for polygon in (free[0] if free else []):
        xs, ys = [int(p[0]) for p in polygon], [int(p[1]) for p in polygon]
        boxes.append((min(xs), min(ys), max(xs), max(ys)))
    width, height = image.size
    cleaned = []
    for x1, y1, x2, y2 in boxes:
        x1, y1 = max(0, x1-BOX_PADDING), max(0, y1-BOX_PADDING)
        x2, y2 = min(width, x2+BOX_PADDING), min(height, y2+BOX_PADDING)
        if x2-x1 >= MIN_BOX_SIDE and y2-y1 >= MIN_BOX_SIDE:
            cleaned.append((x1, y1, x2, y2))
    return sorted(cleaned, key=lambda b: (b[1] // 20, b[0]))

def recognize_crops(crops):
    results = []
    for start in range(0, len(crops), RECOGNITION_BATCH):
        batch = crops[start:start+RECOGNITION_BATCH]
        try:
            texts, probs = recognizer.predict_batch(batch, return_prob=True)
        except Exception:
            pairs = [recognizer.predict(crop, return_prob=True) for crop in batch]
            texts, probs = zip(*pairs) if pairs else ([], [])
        results.extend(zip(texts, probs))
    return results

def ocr_keyframe(image_path):
    image = Image.open(image_path).convert('RGB')
    boxes = detect_boxes(image)
    crops = [image.crop(box) for box in boxes]
    detections = []
    for box, (text, confidence) in zip(boxes, recognize_crops(crops)):
        text, confidence = (text or '').strip(), float(confidence)
        if text and confidence >= MIN_CONFIDENCE:
            detections.append({'text': text, 'confidence': round(confidence, 4), 'box': list(box)})
    return detections

In [ ]:
def output_json_path(video_id): return OUTPUT_ROOT / f'{video_id}.json'
def partial_json_path(video_id): return OUTPUT_ROOT / f'{video_id}.partial.json'

def atomic_write(path, payload):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def ocr_video(video_dir, progress_every=100):
    video_id, mapping = video_dir.name, load_keyframe_map(video_dir.name)
    images = sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS), key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name))
    partial = partial_json_path(video_id)
    if partial.exists() and not OVERWRITE:
        payload = json.loads(partial.read_text(encoding='utf-8'))
        if payload.get('model') != MODEL_ID: payload = None
    else: payload = None
    if payload is None:
        payload = {'video_id': video_id, 'source': str(video_dir), 'model': MODEL_ID, 'language': 'vi', 'min_confidence': MIN_CONFIDENCE, 'has_keyframe_map': mapping is not None, 'complete': False, 'keyframe_count': 0, 'keyframes': []}
    done = {item['keyframe'] for item in payload['keyframes']}
    for index, image_path in enumerate(images, 1):
        if image_path.name in done: continue
        order = keyframe_order(image_path)
        mapped = mapping.get(order) if mapping and order is not None else None
        detections = ocr_keyframe(image_path)
        payload['keyframes'].append({'keyframe': image_path.name, 'n': order, 'frame_idx': mapped['frame_idx'] if mapped else None, 'pts_time': mapped['pts_time'] if mapped else None, 'fps': mapped['fps'] if mapped else None, 'text': ' '.join(d['text'] for d in detections), 'detections': detections})
        payload['keyframe_count'] = len(payload['keyframes'])
        atomic_write(partial, payload)
        if progress_every and index % progress_every == 0: print(f'    {index}/{len(images)} keyframe')
    payload['complete'] = True
    destination = output_json_path(video_id)
    atomic_write(destination, payload)
    if partial.exists(): partial.unlink()
    return payload, destination

## Chạy thử một video

In [ ]:
assert video_dirs, 'Không tìm thấy video nào'
sample_payload, sample_path = ocr_video(video_dirs[0])
print('Đã lưu:', sample_path)
for item in [k for k in sample_payload['keyframes'] if k['text']][:15]:
    print(item['keyframe'], 'frame_idx=', item['frame_idx'], '->', item['text'][:150])

## Xem bounding box (kiểm tra bằng mắt)

Chạy sau cell thử một video ở trên. Lấy `PREVIEW_COUNT` keyframe nhiều chữ nhất của video vừa OCR,
vẽ box lên ảnh rồi hiện inline + lưu ra `/content/ocr_preview/` (bộ nhớ tạm Colab, **không** ghi vào Drive).

Box **xanh lá** = giữ lại. Box **đỏ** = detect được chữ nhưng `confidence < MIN_CONFIDENCE` nên bị loại —
nếu thấy nhiều chữ thật bị đỏ thì hạ `MIN_CONFIDENCE` ở cell cấu hình rồi chạy lại.

Vì file JSON chỉ lưu box đã qua ngưỡng, cell này **OCR lại** đúng vài ảnh preview để lấy cả box bị loại.
Tốn thêm vài giây, không ảnh hưởng kết quả đã ghi.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import ImageDraw, ImageFont

PREVIEW_COUNT = 6      # số keyframe muốn xem
SHOW_DROPPED = True    # vẽ luôn box bị MIN_CONFIDENCE loại (màu đỏ)

PREVIEW_DIR = Path('/content/ocr_preview')   # tạm của Colab, không phải Drive
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# DejaVu Sans đi kèm matplotlib, có đủ dấu tiếng Việt — font mặc định của PIL thì không.
_font_path = font_manager.findfont('DejaVu Sans')


def ocr_keyframe_verbose(image_path):
    """Như ocr_keyframe nhưng trả về CẢ box bị ngưỡng loại, để vẽ preview."""
    image = Image.open(image_path).convert('RGB')
    boxes = detect_boxes(image)
    pairs = recognize_crops([image.crop(box) for box in boxes]) if boxes else []
    detections = []
    for box, (text, confidence) in zip(boxes, pairs):
        text, confidence = (text or '').strip(), float(confidence)
        detections.append({'text': text, 'confidence': confidence, 'box': list(box),
                           'kept': bool(text) and confidence >= MIN_CONFIDENCE})
    return image, detections


def draw_detections(image, detections, show_dropped=SHOW_DROPPED):
    """Vẽ bounding box + text lên bản copy của ảnh, trả về ảnh PIL mới."""
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.truetype(_font_path, max(13, canvas.height // 45))
    line_width = max(2, canvas.height // 400)

    for d in detections:
        if not d['kept'] and not (show_dropped and d['text']):
            continue
        x_min, y_min, x_max, y_max = d['box']
        color = (0, 255, 0) if d['kept'] else (255, 60, 60)
        draw.rectangle([x_min, y_min, x_max, y_max], outline=color,
                       width=line_width if d['kept'] else max(1, line_width - 1))

        label = f"{d['text'] or '?'} ({d['confidence']:.2f})"
        left, top, right, bottom = draw.textbbox((0, 0), label, font=font)
        label_w, label_h = right - left, bottom - top
        # Nhãn đặt phía trên box; nếu sát mép trên thì lật xuống dưới box.
        label_y = y_min - label_h - 3
        if label_y < 0:
            label_y = min(y_max + 2, canvas.height - label_h - 1)
        label_x = min(x_min, max(0, canvas.width - label_w - 2))
        draw.rectangle([label_x, label_y, label_x + label_w + 3, label_y + label_h + 3], fill=color)
        draw.text((label_x + 2, label_y + 1), label, fill=(0, 0, 0), font=font)
    return canvas


preview_video_dir = Path(sample_payload['source'])
with_text = sorted((k for k in sample_payload['keyframes'] if k['text']),
                   key=lambda k: -len(k['detections']))[:PREVIEW_COUNT]

if not with_text:
    print(f"{sample_payload['video_id']}: không keyframe nào có chữ — "
          'thử video khác (đổi video_dirs[0]) hoặc hạ MIN_CONFIDENCE.')
else:
    for item in with_text:
        image_path = preview_video_dir / item['keyframe']
        if not image_path.is_file():
            print('Không thấy ảnh:', image_path)
            continue
        image, detections = ocr_keyframe_verbose(image_path)
        annotated = draw_detections(image, detections)
        saved = PREVIEW_DIR / f"{sample_payload['video_id']}_{image_path.stem}.jpg"
        annotated.save(saved, quality=92)

        n_kept = sum(d['kept'] for d in detections)
        n_dropped = sum(bool(d['text']) and not d['kept'] for d in detections)
        plt.figure(figsize=(16, 16 * annotated.height / annotated.width))
        plt.imshow(annotated)
        plt.title(f"{sample_payload['video_id']} / {item['keyframe']}  "
                  f"(frame_idx={item['frame_idx']}) — {n_kept} box giữ lại, {n_dropped} bị loại",
                  fontsize=11)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print('Text:', ' '.join(d['text'] for d in detections if d['kept']) or '(rỗng)')
        print('Đã lưu:', saved, '\n')

    print(f'Ảnh đã vẽ box nằm ở: {PREVIEW_DIR}')
    print('Tải về: mở tab Files (icon thư mục bên trái) → ocr_preview/ → chuột phải → Download')

## Chạy toàn bộ

In [ ]:
import traceback
MAX_VIDEOS = 0  #@param {type:'integer'}
selected = video_dirs[:MAX_VIDEOS] if MAX_VIDEOS > 0 else video_dirs
success = skipped = failed = 0
failures = []
for index, video_dir in enumerate(selected, 1):
    destination = output_json_path(video_dir.name)
    if destination.exists() and not OVERWRITE:
        try: existing = json.loads(destination.read_text(encoding='utf-8'))
        except Exception: existing = {}
        if existing.get('model') == MODEL_ID and existing.get('complete'):
            skipped += 1; print(f'[{index}/{len(selected)}] SKIP {video_dir.name}'); continue
    print(f'[{index}/{len(selected)}] OCR  {video_dir.name}')
    try:
        payload, _ = ocr_video(video_dir)
        print(f"    {sum(bool(k['text']) for k in payload['keyframes'])}/{payload['keyframe_count']} ảnh có chữ")
        success += 1
    except Exception as exc:
        failed += 1; failures.append({'video_id': video_dir.name, 'error': repr(exc)}); traceback.print_exc()
    finally:
        if use_gpu: torch.cuda.empty_cache()
atomic_write(OUTPUT_ROOT / '_failed.json', failures)
print(f'Hoàn tất: success={success}, skipped={skipped}, failed={failed}')

## Xuất JSONL để index

In [ ]:
jsonl_path = OUTPUT_ROOT / 'ocr_index.jsonl'
output_lines = no_frame_idx = 0
with jsonl_path.open('w', encoding='utf-8') as handle:
    for json_path in sorted(OUTPUT_ROOT.glob('L*_V*.json')):
        payload = json.loads(json_path.read_text(encoding='utf-8'))
        if payload.get('model') != MODEL_ID or not payload.get('complete'): continue
        for keyframe in payload['keyframes']:
            if not keyframe['text']: continue
            no_frame_idx += keyframe['frame_idx'] is None
            row = {'video_id': payload['video_id'], 'frame_idx': keyframe['frame_idx'], 'pts_time': keyframe['pts_time'], 'keyframe': keyframe['keyframe'], 'text': keyframe['text']}
            handle.write(json.dumps(row, ensure_ascii=False) + '\n'); output_lines += 1
print(f'Đã ghi {output_lines} dòng vào {jsonl_path}')
if no_frame_idx: print(f'CẢNH BÁO: {no_frame_idx} dòng thiếu frame_idx')